# F05 - Train, validation, and test splits

**Hardware:** CPU-only · **Time:** 2-3 hours · **Prerequisite:** F04

A split is a promise: the test set is untouched until the final evaluation. This
notebook builds three leakage-free splits for a synthetic house-price table,
then demonstrates two common leaks: a feature computed from the target and a
preprocessing step fitted before the split.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

SEED = 42
rng = np.random.default_rng(SEED)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Build a table with groups and a target

Each row is a house. `neighbourhood` is a group: houses in the same
neighbourhood share unmeasured location effects. A random row split can put
near-identical houses on both sides, which inflates the score.

In [2]:
def make_houses(n: int = 600, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    area = rng.normal(1800, 500, n).clip(600, 4000)
    bedrooms = rng.integers(1, 6, n)
    age = rng.integers(0, 60, n)
    distance = rng.gamma(2, 2, n).clip(0.5, 25)
    garage = rng.integers(0, 2, n)
    hood = rng.choice(["north", "south", "east", "west"], n, p=[0.3, 0.25, 0.25, 0.2])
    effect = pd.Series(hood).map({"north": 35000, "south": -10000, "east": 15000, "west": -5000}).to_numpy()
    price = (
        120000 + 95 * area + 18000 * bedrooms - 900 * age - 4500 * distance
        + 12000 * garage + 0.00004 * area**2 + effect + rng.normal(0, 20000, n)
    ).clip(50000, None)
    return pd.DataFrame(
        {
            "area_sqft": area.round(1),
            "bedrooms": bedrooms,
            "age_years": age,
            "distance_km": distance.round(2),
            "garage": garage,
            "neighbourhood": hood,
            "sale_price": price.round(0),
        }
    )


houses = make_houses()
print("shape:", houses.shape)
print(houses.head(3).to_string(index=False))
assert houses.shape == (600, 7)
assert houses["sale_price"].min() > 0

shape: (600, 7)
 area_sqft  bedrooms  age_years  distance_km  garage neighbourhood  sale_price
    1952.4         2          4         2.98       1          west    337153.0
    1280.0         3         15         6.67       1          west    249256.0
    2175.2         1         27         5.67       0         north    341102.0


## 2. A 60/20/20 split

First hold out 20% as the final test set. Then split the remaining 80% into
training and validation. The validation set guides model choices; the test set
answers "how well does the final choice generalize?"

In [3]:
train_val, test = train_test_split(houses, test_size=0.20, random_state=SEED)
train, validation = train_test_split(train_val, test_size=0.25, random_state=SEED)
print("train:", train.shape, "validation:", validation.shape, "test:", test.shape)
print("fractions:", round(len(train) / len(houses), 2), round(len(validation) / len(houses), 2), round(len(test) / len(houses), 2))
assert len(train) + len(validation) + len(test) == len(houses)
assert abs(len(test) / len(houses) - 0.20) < 0.01

train: (360, 7) validation: (120, 7) test: (120, 7)
fractions: 0.6 0.2 0.2


## 3. Stratify a regression target

Stratification preserves the distribution of a variable across splits. For a
continuous target, bin it first. The code below bins `sale_price` into quartiles
and checks that the test set has a similar mix. Do not stratify on the target
when that would leak information about individual outcomes in a way that changes
the task; here the goal is only to preserve the price distribution.

In [4]:
houses["price_band"] = pd.qcut(houses["sale_price"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
train_s, test_s = train_test_split(
    houses, test_size=0.20, random_state=SEED, stratify=houses["price_band"]
)
print("overall bands:", houses["price_band"].value_counts(normalize=True).round(2).to_dict())
print("test bands:", test_s["price_band"].value_counts(normalize=True).round(2).to_dict())
assert set(train_s["price_band"]) == set(test_s["price_band"])

overall bands: {'Q1': 0.25, 'Q2': 0.25, 'Q3': 0.25, 'Q4': 0.25}
test bands: {'Q1': 0.25, 'Q2': 0.25, 'Q3': 0.25, 'Q4': 0.25}


## 4. Group-aware splitting

If a group can appear in both training and test data, the model may memorize
group-specific effects. `GroupShuffleSplit` keeps each neighbourhood entirely on
one side. The trade-off is that the test set may not represent every group.

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(houses, groups=houses["neighbourhood"]))
group_train, group_test = houses.iloc[train_idx], houses.iloc[test_idx]
print("train groups:", sorted(group_train["neighbourhood"].unique()))
print("test groups:", sorted(group_test["neighbourhood"].unique()))
print("group overlap:", set(group_train["neighbourhood"]) & set(group_test["neighbourhood"]))
assert not set(group_train["neighbourhood"]) & set(group_test["neighbourhood"])
assert len(group_train) + len(group_test) == len(houses)

train groups: ['east', 'south', 'west']
test groups: ['north']
group overlap: set()


## 5. Leakage demonstration 1: a target-derived feature

`price_per_city_mean` is computed from `sale_price` on the full table. It looks
like a useful location feature, but it contains the answer. A model with this
feature scores far too well. After removing it, the score drops to a believable
level.

In [6]:
leaky = houses.copy()
leaky["price_vs_city_mean"] = leaky["sale_price"] - leaky["sale_price"].mean()
X_leaky = leaky[["area_sqft", "bedrooms", "price_vs_city_mean"]]
X_clean = leaky[["area_sqft", "bedrooms"]]
y = leaky["sale_price"]

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=SEED)
leaky_model = LinearRegression().fit(X_train, y_train)
leaky_score = r2_score(y_test, leaky_model.predict(X_test))

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y, test_size=0.2, random_state=SEED)
clean_model = LinearRegression().fit(X_train_c, y_train_c)
clean_score = r2_score(y_test_c, clean_model.predict(X_test_c))

print("R2 with target-derived feature:", round(leaky_score, 3))
print("R2 without it:", round(clean_score, 3))
assert leaky_score > 0.95
assert clean_score < leaky_score

R2 with target-derived feature: 1.0
R2 without it: 0.703


## 6. Leakage demonstration 2: scaling before the split

Fitting a scaler on all rows lets the training process see the test set's mean
and variance. The effect is small here, but the principle is not: every learned
transformation belongs inside a pipeline fitted on training data only.

In [7]:
feature_columns = ["area_sqft", "bedrooms", "age_years", "distance_km", "garage"]
X = houses[feature_columns].to_numpy(dtype=float)
y = houses["sale_price"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
scaler_all = StandardScaler().fit(X)
X_train_leaky = scaler_all.transform(X_train)
X_test_leaky = scaler_all.transform(X_test)

scaler_train = StandardScaler().fit(X_train)
X_train_clean = scaler_train.transform(X_train)
X_test_clean = scaler_train.transform(X_test)

leaky = r2_score(y_test, LinearRegression().fit(X_train_leaky, y_train).predict(X_test_leaky))
clean = r2_score(y_test, LinearRegression().fit(X_train_clean, y_train).predict(X_test_clean))
print("test-set mean seen by scaler (leaky):", round(float(X_test.mean()), 3))
print("R2, scaler fit on all data:", round(leaky, 4))
print("R2, scaler fit on train only:", round(clean, 4))
assert abs(leaky - clean) < 0.05

test-set mean seen by scaler (leaky): 369.463
R2, scaler fit on all data: 0.7976
R2, scaler fit on train only: 0.7976


## 7. Duplicate leakage

If the same house appears twice and one copy lands in training while the other
lands in test, the model can memorize it. Always deduplicate before splitting,
using a stable identifier or a content hash.

In [8]:
with_duplicates = pd.concat([houses, houses.sample(20, random_state=SEED)], ignore_index=True)
dup_train, dup_test = train_test_split(with_duplicates, test_size=0.2, random_state=SEED)
overlap = len(pd.merge(dup_train, dup_test, on=list(houses.columns), how="inner"))
print("duplicate rows before split:", int(with_duplicates.duplicated().sum()))
print("cross-split duplicate matches:", overlap)

deduped = with_duplicates.drop_duplicates()
clean_train, clean_test = train_test_split(deduped, test_size=0.2, random_state=SEED)
clean_overlap = len(pd.merge(clean_train, clean_test, on=list(houses.columns), how="inner"))
print("cross-split matches after deduplication:", clean_overlap)
assert overlap > 0
assert clean_overlap == 0

duplicate rows before split: 20
cross-split duplicate matches: 11
cross-split matches after deduplication: 0


## Check your work

- [ ] You can explain why the test set is held out until the end.
- [ ] You created a 60/20/20 split and a group-aware split.
- [ ] You identified a target-derived feature and removed it.
- [ ] You fit the scaler on training data only.
- [ ] You removed duplicates before splitting.

**Next:** F06 - Exploratory data analysis and visualization.